In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv("Loan Prediction.csv")

In [ ]:
df.head()

In [ ]:
df

In [ ]:
df.drop('Id' , axis = 1, inplace = True)

In [ ]:
df

In [ ]:
df['Profession'].value_counts()

In [ ]:
df['House_Ownership'].value_counts()

In [ ]:
df.corr(numeric_only=True)

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

In [ ]:
df.head()

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder()

In [ ]:
df['Married/Single'] = le.fit_transform(df['Married/Single'])
df['Car_Ownership'] = le.fit_transform(df['Car_Ownership'])
df

In [ ]:
df.head()

In [ ]:
df

In [ ]:
top_professions = df['Profession'].value_counts().index[:15]
df['Profession'] = df['Profession'].apply(lambda x: x if x in top_professions else 'Other')
df = pd.get_dummies(df, columns=['Profession'], drop_first=True, dtype=int)

In [ ]:
df

In [ ]:
sns.heatmap(df.corr(numeric_only=True), annot=False, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap of Numeric Features')
plt.show()

In [ ]:
sns.boxplot(data=df, x='Profession_Chemical_engineer', y='Income')
plt.xticks(rotation=90)

In [ ]:
df['CITY'].value_counts()

In [ ]:
df.isnull().sum()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
ss = StandardScaler()

In [ ]:
df.columns

In [ ]:
ss_columns = ['Income','Age','Experience','CURRENT_JOB_YRS','CURRENT_HOUSE_YRS']

In [ ]:
df[ss_columns] = ss.fit_transform(df[ss_columns])

In [ ]:
df

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe = OneHotEncoder()

In [ ]:
df

In [ ]:
df['House_Ownership'].value_counts()

In [ ]:
df['CITY'].value_counts()

In [ ]:
df.drop('CITY' , axis = 1 , inplace = True)

In [ ]:
df.head()

In [ ]:
df['STATE'].value_counts()

In [ ]:
df['STATE'] = df['STATE'].str.replace(r'\[.*\]', '', regex=True).str.strip()


In [ ]:
df['STATE'].value_counts()

In [ ]:
df = pd.get_dummies(df , columns = ['STATE'] , drop_first = True , dtype = int)

In [ ]:
df

In [ ]:
df.select_dtypes(include=['object', 'category']).columns

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train , x_test , y_train , y_test = train_test_split(df.drop('Risk_Flag' , axis = 1) , df['Risk_Flag'] , test_size = 0.2 , random_state = 42)

In [ ]:
x_train.shape

In [ ]:
x_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model_lr = LogisticRegression(class_weight= 'balanced')

In [ ]:
le_house = LabelEncoder()
x_train['House_Ownership'] = le_house.fit_transform(x_train['House_Ownership'])
x_test['House_Ownership'] = le_house.transform(x_test['House_Ownership'])

model_lr.fit(x_train , y_train)

In [ ]:
y_pred_lr = model_lr.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report , recall_score , precision_score , f1_score

In [ ]:
accuracy_lr = accuracy_score(y_test , y_pred_lr)

In [ ]:
accuracy_lr

In [ ]:
confusion_matrix(y_test , y_pred_lr)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
model_dt = DecisionTreeClassifier(class_weight= 'balanced')

In [ ]:
model_dt.fit(x_train , y_train)

In [ ]:
y_pred_dt = model_dt.predict(x_test)

In [ ]:
accuracy_dt = accuracy_score(y_test , y_pred_dt)

In [ ]:
accuracy_dt

In [ ]:
confusion_matrix(y_test , y_pred_dt)

In [ ]:
recall_dt = recall_score(y_test , y_pred_dt)

In [ ]:
recall_dt

In [ ]:
precision_score(y_test , y_pred_dt)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
model_knn = KNeighborsClassifier(n_neighbors=5, n_jobs = -1)

In [ ]:
model_knn.fit(x_train , y_train)

In [ ]:
import numpy as np

total_samples = len(x_test)
batch_size = 1000

batch_accuracies = []

for i in range(0, total_samples, batch_size):

    x_batch = x_test.iloc[i : i + batch_size]
    y_batch = y_test.iloc[i : i + batch_size]


    y_pred_batch = model_knn.predict(x_batch)
    acc = accuracy_score(y_batch, y_pred_batch)

    batch_accuracies.append(acc)

    print(f"Batch {i//batch_size + 1}: Samples {i} to {i + len(x_batch)} | Accuracy: {acc:.4f}")

avg_accuracy = np.mean(batch_accuracies)

print("\n" + "="*40)
print(f"Final Average KNN Accuracy: {avg_accuracy:.4f}")
print("="*40)

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

print("Evaluating all metrics (it might take a few seconds)...")

y_pred_all = cross_val_predict(model_knn, x_test, y_test, cv=5, n_jobs=-1)

print("\n" + "="*50)
print("              KNN CLASSIFICATION REPORT            ")
print("="*50)
print(classification_report(y_test, y_pred_all))
print("="*50)

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import classification_report

print("Evaluating Decision Tree metrics...")

y_pred_dt_all = cross_val_predict(model_dt, x_test, y_test, cv=5, n_jobs=-1)

print("\n" + "="*50)
print("          DECISION TREE CLASSIFICATION REPORT      ")
print("="*50)
print(classification_report(y_test, y_pred_dt_all))
print("="*50)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
model_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', n_jobs=-1 , random_state = 42)

In [ ]:
model_rf.fit(x_train , y_train)

In [ ]:
y_pred_rf = model_rf.predict(x_test)

In [ ]:
print(classification_report(y_test , y_pred_rf))

In [ ]:
from xgboost import XGBClassifier

In [ ]:
model_xgb = XGBClassifier(n_estimators=100,scale_pos_weight = 7,random_state = 42, n_jobs=-1)

In [ ]:
model_xgb.fit(x_train , y_train)

In [ ]:
y_pred_xgb = model_xgb.predict(x_test)

In [ ]:
print(classification_report(y_test , y_pred_xgb))